In [ ]:
import os
import shutil
import json
from datetime import datetime
from kaggle.api.kaggle_api_extended import KaggleApi

def download_and_prepare():
    api = KaggleApi()
    api.authenticate()

    dataset = "olistbr/brazilian-ecommerce"
    base_path = r"C:\Mini Proyek\PROJECT-OLIST-PIPELINE\data"
    raw_path = os.path.join(base_path, "raw")
    checkpoint_file = os.path.join(base_path, "download_checkpoint.json")

    # Cek checkpoint - apakah sudah download sebelumnya?
    if os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'r') as f:
                checkpoint_data = json.load(f)
            
            download_time = checkpoint_data.get('download_time')
            dataset_name = checkpoint_data.get('dataset')
            
            print(f"[CHECKPOINT] Dataset sudah didownload sebelumnya pada: {download_time}")
            print(f"[CHECKPOINT] Dataset: {dataset_name}")
            print("[SKIP] Melewati proses download...")
            
            # Langsung ke proses organize file
            organize_files(base_path, raw_path)
            return
            
        except Exception as e:
            print(f"[WARNING] Error membaca checkpoint: {e}")
            print("[CONTINUE] Melanjutkan proses download...")

    os.makedirs(raw_path, exist_ok=True)

    # Download
    print(f"[DOWNLOAD] Memulai download dataset: {dataset}")
    api.dataset_download_files(dataset, path=base_path, unzip=True)

    print('[DONE] Download finished:', base_path, flush=True)

    # Buat checkpoint setelah download berhasil
    checkpoint_data = {
        'dataset': dataset,
        'download_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'status': 'completed',
        'base_path': base_path
    }
    
    with open(checkpoint_file, 'w') as f:
        json.dump(checkpoint_data, f, indent=2)
    
    print(f"[CHECKPOINT] Checkpoint dibuat: {checkpoint_file}")

    # Organize files
    organize_files(base_path, raw_path)

def organize_files(base_path, raw_path):
    """Fungsi terpisah untuk organize files"""
    
    # Mapping dataset
    mapping = {
        "orders": "olist_orders_dataset.csv",
        "customers": "olist_customers_dataset.csv",
        "payments": "olist_order_payments_dataset.csv",
        "items": "olist_order_items_dataset.csv",
        "geolocation": "olist_geolocation_dataset.csv",
        "order_review": "olist_order_reviews_dataset.csv",
        "products": "olist_products_dataset.csv",
        "sellers": "olist_sellers_dataset.csv",
        "product_category": "product_category_name_translation.csv"
    }

    # Move file ke folder masing-masing
    for folder, filename in mapping.items():
        folder_path = os.path.join(raw_path, folder)
        os.makedirs(folder_path, exist_ok=True)

        src = os.path.join(base_path, filename)
        dst = os.path.join(folder_path, filename)

        if os.path.exists(src):
            shutil.move(src, dst)
            print(f"[MOVE] {filename} -> {folder}/")
        else:
            print(f"[WARNING] File tidak ditemukan: {filename}")

    print('[DONE] File dataset sudah dirapikan:', base_path, flush=True)

if __name__ == "__main__":
    download_and_prepare()

[DOWNLOAD] Memulai download dataset: olistbr/brazilian-ecommerce
Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
[DONE] Download finished: C:\Mini Proyek\PROJECT-OLIST-PIPELINE\data
[CHECKPOINT] Checkpoint dibuat: C:\Mini Proyek\PROJECT-OLIST-PIPELINE\data\download_checkpoint.json
[MOVE] olist_orders_dataset.csv -> orders/
[MOVE] olist_customers_dataset.csv -> customers/
[MOVE] olist_order_payments_dataset.csv -> payments/
[MOVE] olist_order_items_dataset.csv -> items/
[MOVE] olist_geolocation_dataset.csv -> geolocation/
[MOVE] olist_order_reviews_dataset.csv -> order_review/
[MOVE] olist_products_dataset.csv -> products/
[MOVE] olist_sellers_dataset.csv -> sellers/
[MOVE] product_category_name_translation.csv -> product_category/
[DONE] File dataset sudah dirapikan: C:\Mini Proyek\PROJECT-OLIST-PIPELINE\data


In [ ]:
def validate_download_structure():
    """Validasi struktur folder dan jumlah file yang didownload"""
    
    base_path = r"C:\Mini Proyek\PROJECT-OLIST-PIPELINE\data"
    raw_path = os.path.join(base_path, "raw")
    
    # Expected folder dan file
    expected_folders = [
        "orders", "customers", "payments", "items", 
        "geolocation", "order_review", "products", "sellers", "product_category"
    ]
    
    expected_files = {
        "orders": "olist_orders_dataset.csv",
        "customers": "olist_customers_dataset.csv",
        "payments": "olist_order_payments_dataset.csv",
        "items": "olist_order_items_dataset.csv",
        "geolocation": "olist_geolocation_dataset.csv",
        "order_review": "olist_order_reviews_dataset.csv",
        "products": "olist_products_dataset.csv",
        "sellers": "olist_sellers_dataset.csv",
        "product_category": "product_category_name_translation.csv"
    }
    
    print("="*80)
    print("[VALIDATION] Struktur Folder & File Download")
    print("="*80)
    print(f"\nBase Path: {base_path}")
    print(f"Raw Path: {raw_path}\n")
    
    total_files = 0
    downloaded_files = 0
    missing_files = []
    
    # Validasi setiap folder dan file
    for folder in expected_folders:
        folder_path = os.path.join(raw_path, folder)
        file_name = expected_files[folder]
        file_path = os.path.join(folder_path, file_name)
        
        folder_exists = os.path.exists(folder_path)
        file_exists = os.path.exists(file_path)
        
        status = "✓" if file_exists else "✗"
        total_files += 1
        
        if file_exists:
            downloaded_files += 1
            print(f"{status} {folder:20} -> {file_name}")
        else:
            print(f"{status} {folder:20} -> {file_name} [MISSING]")
            missing_files.append(folder)
    
    # Summary
    print("\n" + "="*80)
    print(f"[SUMMARY] Total Files: {downloaded_files}/{total_files} downloaded")
    print("="*80)
    
    if missing_files:
        print(f"\n  Missing files in folders: {', '.join(missing_files)}")
    else:
        print(f"\n✓ Semua file berhasil didownload!")
    
    return downloaded_files == total_files

# Jalankan validasi
result = validate_download_structure()

[VALIDATION] Struktur Folder & File Download

Base Path: C:\Mini Proyek\PROJECT-OLIST-PIPELINE\data
Raw Path: C:\Mini Proyek\PROJECT-OLIST-PIPELINE\data\raw

✓ orders               -> olist_orders_dataset.csv
✓ customers            -> olist_customers_dataset.csv
✓ payments             -> olist_order_payments_dataset.csv
✓ items                -> olist_order_items_dataset.csv
✓ geolocation          -> olist_geolocation_dataset.csv
✓ order_review         -> olist_order_reviews_dataset.csv
✓ products             -> olist_products_dataset.csv
✓ sellers              -> olist_sellers_dataset.csv
✓ product_category     -> product_category_name_translation.csv

[SUMMARY] Total Files: 9/9 downloaded

✓ Semua file berhasil didownload!
